# KRX 종목 중 Google 뉴스 언급 빈도 상위 N종목 추출

한국거래소(KRX) 상장 종목 중 Google 뉴스에서 가장 빈번하게 거론되는 종목을 추출합니다.

**동작 방식**
1. KRX 전체 상장 종목 리스트(종목명, 티커)를 가져온다. (`FinanceDataReader` 사용)
2. 여러 검색 키워드로 Google 뉴스 RSS를 조회하되, 검색어에 `after:YYYY-MM-DD before:YYYY-MM-DD` 연산자를 붙여 지정한 기간의 기사만 가져온다. (RSS의 실제 발행일로 한 번 더 필터링)
3. 수집한 기사 텍스트 안에서 각 종목명이 몇 번 등장하는지 카운트한다.
4. 언급 횟수 기준으로 내림차순 정렬하여 상위 N종목을 산출한다.
5. 결과를 엑셀(.xlsx) 파일로 저장한다.

**주의**
- 이 방식은 "구글 트렌드/토픽"의 공식 순위 API가 아니라, 구글 뉴스 검색 결과를 기반으로 한 근사치입니다.
- 종목명이 너무 짧으면(예: "한화", "SK" 등) 일반명사와 겹쳐 오탐이 생길 수 있으므로 `MIN_NAME_LEN` 값으로 필터링합니다.
- 네트워크 호출량이 많으므로 요청 사이에 딜레이를 둡니다.

## 0. 패키지 설치
필요 시 아래 셀을 한 번 실행하세요.

In [ ]:
%pip install FinanceDataReader feedparser requests pandas openpyxl --quiet

## 1. 라이브러리 임포트

In [1]:
import re
import time
from datetime import datetime, date
from collections import Counter

import requests
import feedparser
import pandas as pd
import FinanceDataReader as fdr

## 2. 설정값

아래 변수들을 원하는 값으로 수정한 뒤 실행하세요.

- `START_DATE`, `END_DATE`: 조회 기간 (`YYYY-MM-DD` 형식 문자열, `None`이면 기간 제한 없음)
- `TOP_N`: 추출할 종목 수
- `MIN_NAME_LEN`: 오탐 방지를 위한 최소 종목명 길이
- `OUTPUT_PATH`: 결과를 저장할 엑셀 파일 경로

In [2]:
# ------------------------------------------------------------------
# 설정값 - 필요에 따라 수정하세요
# ------------------------------------------------------------------
START_DATE = "2026-06-01"   # 조회 시작일 (YYYY-MM-DD) 또는 None
END_DATE = "2026-06-30"     # 조회 종료일 (YYYY-MM-DD) 또는 None
TOP_N = 20                  # 추출할 종목 수
MIN_NAME_LEN = 2            # 오탐 방지를 위한 최소 종목명 길이
OUTPUT_PATH = "krx_google_trending_top20.xlsx"  # 결과 저장 경로

REQUEST_DELAY = 1.0            # 요청 간 딜레이(초) - 과도한 요청 방지
MAX_ARTICLES_PER_QUERY = 100   # 쿼리당 최대 수집 기사 수 (RSS 특성상 대략치)

# Google 뉴스에서 시장 전반의 기사를 폭넓게 긁어오기 위한 검색어들
SEARCH_QUERIES = [
    "코스피",
    "코스닥",
    "증시",
    "주식시장",
    "상한가",
    "실적발표",
    "주가",
]

# 날짜 형식 간단 검증
for label, value in (("START_DATE", START_DATE), ("END_DATE", END_DATE)):
    if value:
        datetime.strptime(value, "%Y-%m-%d")  # 형식이 틀리면 여기서 에러 발생

if START_DATE and END_DATE and START_DATE > END_DATE:
    raise ValueError("START_DATE는 END_DATE보다 이전이어야 합니다.")

## 3. KRX 상장 종목 리스트 수집

In [3]:
def get_krx_stock_list(min_name_len):
    """KRX 전체 상장 종목(코스피+코스닥) 목록을 가져온다."""
    print("[1/4] KRX 상장 종목 리스트 수집 중...")
    df = fdr.StockListing("KRX")  # 종목명, 코드 등 포함
    name_col = "Name" if "Name" in df.columns else "name"
    code_col = "Code" if "Code" in df.columns else "code"

    stocks = df[[code_col, name_col]].dropna()
    stocks = stocks[stocks[name_col].str.len() >= min_name_len]
    stock_list = list(zip(stocks[code_col], stocks[name_col]))
    print(f"  -> 총 {len(stock_list)}개 종목 확보")
    return stock_list


stock_list = get_krx_stock_list(MIN_NAME_LEN)
stock_list[:10]  # 확인용 미리보기

[1/4] KRX 상장 종목 리스트 수집 중...
  -> 총 2873개 종목 확보


[('005930', '삼성전자'),
 ('000660', 'SK하이닉스'),
 ('402340', 'SK스퀘어'),
 ('009150', '삼성전기'),
 ('005935', '삼성전자우'),
 ('005380', '현대차'),
 ('373220', 'LG에너지솔루션'),
 ('032830', '삼성생명'),
 ('028260', '삼성물산'),
 ('207940', '삼성바이오로직스')]

## 4. Google 뉴스 기사 수집 (기간 필터 적용)

In [4]:
def build_query(base_query, start, end):
    """검색어에 기간 필터(after/before)를 덧붙인다."""
    q = base_query
    if start:
        q += f" after:{start}"
    if end:
        q += f" before:{end}"
    return q


def parse_entry_date(entry):
    """RSS 엔트리에서 발행일을 date 객체로 변환한다. 실패 시 None."""
    parsed = getattr(entry, "published_parsed", None)
    if not parsed:
        return None
    try:
        return date(parsed.tm_year, parsed.tm_mon, parsed.tm_mday)
    except Exception:
        return None


def fetch_google_news_titles(query, start, end, lang="ko", country="KR"):
    """Google 뉴스 RSS에서 특정 검색어(+기간)에 대한 기사 제목+요약 리스트를 가져온다."""
    url = (
        f"https://news.google.com/rss/search?q={requests.utils.quote(query)}"
        f"&hl={lang}&gl={country}&ceid={country}:{lang}"
    )
    try:
        resp = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        feed = feedparser.parse(resp.content)
    except Exception as e:
        print(f"  [경고] '{query}' 요청 실패: {e}")
        return []

    start_d = datetime.strptime(start, "%Y-%m-%d").date() if start else None
    end_d = datetime.strptime(end, "%Y-%m-%d").date() if end else None

    texts = []
    for entry in feed.entries[:MAX_ARTICLES_PER_QUERY]:
        entry_date = parse_entry_date(entry)
        if entry_date:
            if start_d and entry_date < start_d:
                continue
            if end_d and entry_date > end_d:
                continue

        title = getattr(entry, "title", "") or ""
        summary = getattr(entry, "summary", "") or ""
        texts.append(f"{title} {summary}")
    return texts


def collect_all_articles(start, end):
    """설정된 검색어(+기간)들로 뉴스 기사 텍스트를 모두 수집한다."""
    print("[2/4] Google 뉴스 기사 수집 중...")
    if start or end:
        print(f"  - 조회 기간: {start or '제한없음'} ~ {end or '제한없음'}")

    all_texts = []
    for base_q in SEARCH_QUERIES:
        query = build_query(base_q, start, end)
        print(f"  - 검색어: {query}")
        texts = fetch_google_news_titles(query, start, end)
        all_texts.extend(texts)
        time.sleep(REQUEST_DELAY)
    print(f"  -> 총 {len(all_texts)}개 기사(제목+요약) 수집")
    return all_texts


articles = collect_all_articles(START_DATE, END_DATE)
len(articles)

[2/4] Google 뉴스 기사 수집 중...
  - 조회 기간: 2026-06-01 ~ 2026-06-30
  - 검색어: 코스피 after:2026-06-01 before:2026-06-30
  - 검색어: 코스닥 after:2026-06-01 before:2026-06-30
  - 검색어: 증시 after:2026-06-01 before:2026-06-30
  - 검색어: 주식시장 after:2026-06-01 before:2026-06-30
  - 검색어: 상한가 after:2026-06-01 before:2026-06-30
  - 검색어: 실적발표 after:2026-06-01 before:2026-06-30
  - 검색어: 주가 after:2026-06-01 before:2026-06-30
  -> 총 700개 기사(제목+요약) 수집


700

## 5. 종목명 언급 빈도 집계

In [5]:
def count_stock_mentions(stock_list, articles):
    """기사 텍스트 안에서 각 종목명 언급 횟수를 카운트한다."""
    print("[3/4] 종목명 언급 빈도 집계 중...")
    counter = Counter()

    # 긴 종목명부터 매칭해서 짧은 이름이 긴 이름의 부분 문자열로 오매칭되는 것을 최소화
    sorted_stocks = sorted(stock_list, key=lambda x: len(x[1]), reverse=True)
    combined_text = "\n".join(articles)

    for code, name in sorted_stocks:
        pattern = re.escape(name)
        count = len(re.findall(pattern, combined_text))
        if count > 0:
            counter[(code, name)] = count

    return counter


if not articles:
    raise RuntimeError("수집된 기사가 없습니다. 네트워크 연결, 검색어, 또는 조회 기간을 확인하세요.")

counter = count_stock_mentions(stock_list, articles)
top_stocks = counter.most_common(TOP_N)

if not top_stocks:
    raise RuntimeError("언급된 종목을 찾지 못했습니다.")

result_df = pd.DataFrame(
    [
        {"순위": rank, "종목명": name, "종목코드": code, "언급 횟수": count}
        for rank, ((code, name), count) in enumerate(top_stocks, start=1)
    ]
)
result_df

[3/4] 종목명 언급 빈도 집계 중...


,순위,종목명,종목코드,언급 횟수
0,1,E1,017940,293
1,2,YW,051390,129
2,3,DB,012030,85
3,4,SG,255220,82
4,5,SK,034730,70
5,6,이닉스,452400,64
6,7,YTN,040300,61
7,8,3S,060310,56
8,9,E8,418620,50
9,10,SK하이닉스,000660,46


## 6. 결과를 엑셀 파일로 저장

In [6]:
def save_to_excel(df, output_path, start, end):
    """결과를 엑셀 파일로 저장한다."""
    print(f"[4/4] 엑셀 파일 저장 중... ({output_path})")

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="결과")

        meta = pd.DataFrame(
            {
                "항목": ["조회 시작일", "조회 종료일", "생성 시각"],
                "값": [
                    start or "제한없음",
                    end or "제한없음",
                    datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                ],
            }
        )
        meta.to_excel(writer, index=False, sheet_name="조회조건")

        worksheet = writer.sheets["결과"]
        for i, col in enumerate(df.columns, start=1):
            max_len = max(df[col].astype(str).map(len).max(), len(col)) + 2
            worksheet.column_dimensions[chr(64 + i)].width = max_len

    print(f"  -> 저장 완료: {output_path}")


save_to_excel(result_df, OUTPUT_PATH, START_DATE, END_DATE)

[4/4] 엑셀 파일 저장 중... (krx_google_trending_top20.xlsx)
  -> 저장 완료: krx_google_trending_top20.xlsx
